# Streaming Chat Completions with DistLLM

This notebook shows how to stream tokens from the DistLLM OpenAI-compatible API **token-by-token** using the official `openai` Python client pointed at `http://localhost:8000/v1`.

Streaming makes long responses feel responsive: text appears as it is generated instead of waiting for the full completion.

## Requirements

```bash
pip install openai distllm_sdk
```

Start the DistLLM API server before running the code cells (see the README).

In [ ]:
# ---------------------------------------------------------------------------
# Connection configuration
# ---------------------------------------------------------------------------
# Point the OpenAI-compatible client at a running DistLLM API.
# Start the server first, e.g.:
#     distllm-api --model distributed-llm --host 0.0.0.0 --port 8000
#
# The API key is not required for a local deployment; set DISTLLM_API_KEY if
# your deployment enforces authentication.
import os

BASE_URL = os.environ.get("DISTLLM_BASE_URL", "http://localhost:8000/v1")
API_KEY = os.environ.get("DISTLLM_API_KEY", "not-needed")
MODEL = os.environ.get("DISTLLM_MODEL", "distributed-llm")

print(f"Using endpoint: {BASE_URL}")
print(f"Model: {MODEL}")


In [ ]:
# ---------------------------------------------------------------------------
# Create the streaming client
# ---------------------------------------------------------------------------
# Prefer the official openai client; fall back to distllm_sdk.compat if missing.
try:
    from openai import OpenAI
    client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
    print('Using official openai client.')
except ImportError:
    from distllm_sdk.compat import openai_compat as oc
    client = oc.OpenAI(base_url=BASE_URL, api_key=API_KEY)
    print('openai client missing -> using distllm_sdk.compat.openai_compat.')


In [ ]:
# ---------------------------------------------------------------------------
# Stream a single response token-by-token
# ---------------------------------------------------------------------------
messages = [
    {'role': 'system', 'content': 'You are a helpful assistant.'},
    {'role': 'user', 'content': 'Write a short haiku about distributed inference.'},
]

print('Streaming response:\n')
try:
    collected = []
    stream = client.chat.completions.create(
        model=MODEL, messages=messages, temperature=0.8, max_tokens=120, stream=True)
    for chunk in stream:
        # Each chunk carries one delta; guard against empty deltas.
        if not chunk.choices:
            continue
        delta = chunk.choices[0].delta
        token = getattr(delta, 'content', None)
        if token:
            print(token, end='', flush=True)
            collected.append(token)
    print('\n\n[done]')
except Exception as exc:  # pragma: no cover
    print(f'\n[!] Streaming request failed: {exc}')
    print('    Is the DistLLM API running at', BASE_URL, '?')


In [ ]:
# ---------------------------------------------------------------------------
# Stream with usage info and a final summary line
# note: usage stream_options are supported by the SDK compat layer.
try:
    final_usage = None
    stream = client.chat.completions.create(
        model=MODEL,
        messages=[{'role': 'user', 'content': 'List three benefits of edge deployment.'}],
        max_tokens=120, stream=True, stream_options={'include_usage': True})
    print('Streaming (with usage):\n')
    for chunk in stream:
        if chunk.choices:
            token = getattr(chunk.choices[0].delta, 'content', None)
            if token:
                print(token, end='', flush=True)
        # The SDK compat layer surfaces a final usage chunk.
        if getattr(chunk, 'usage', None):
            final_usage = chunk.usage
    if final_usage:
        print(f'\n\n[usage] prompt={final_usage.prompt_tokens} '
              f'completion={final_usage.completion_tokens} '
              f'total={final_usage.total_tokens}')
except Exception as exc:  # pragma: no cover
    print(f'\n[!] Streaming (usage) request failed: {exc}')
